# Render cartoon short (three.js + Remotion) — Colab

Jalankan sel satu per satu (Runtime → Run all juga boleh).

- Runtime: **CPU** sudah cukup (Mesa llvmpipe). GPU tidak dipakai Chromium di Colab.
- Kalau `smoke_gl.sh` melaporkan `SIAP RENDER` pada salah satu backend, lanjut render.
- Kualitas: `SCALE=1` = 1080x1920 (cepat), `SCALE=1.3333` = 1440x2560 (hasil akhir).


In [ ]:
# 1. paket sistem: Mesa (llvmpipe) + dependensi Chromium + ffmpeg
!apt-get -qq update
!DEBIAN_FRONTEND=noninteractive apt-get -qq install -y --no-install-recommends \
  libgl1-mesa-dri libglx-mesa0 libegl1 libgles2 libglu1-mesa mesa-utils \
  libnss3 libdbus-1-3 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 \
  libxrandr2 libgbm1 libpango-1.0-0 libcairo2 ffmpeg xz-utils > /dev/null
!glxinfo -B 2>/dev/null | head -6 || echo 'glxinfo butuh X; headless tetap jalan'
!node -v && npm -v

In [ ]:
# 2. ambil proyek + dependensi
%cd /content
!rm -rf cartoon-short
!git clone -q https://github.com/HiuraKiyowoo/cartoon-short.git
%cd /content/cartoon-short
!npm install --no-audit --no-fund 2>&1 | tail -5

In [ ]:
# 3. cari backend GL yang benar-benar menghasilkan gambar (probe vs beerflood)
!bash scripts/smoke_gl.sh

In [ ]:
# 4. render video penuh (ganti GL= hasil langkah 3, mis. GL=angle)
%env SCALE=1
%env GL=angle
!bash scripts/render.sh 2>&1 | tail -20

In [ ]:
# 5. unduh hasilnya
from google.colab import files
import os
p = '/content/cartoon-short/out/final/video.mp4'
print('ada:', os.path.exists(p), (os.path.getsize(p)//1024, 'KB') if os.path.exists(p) else '')
files.download(p)